## Objetivo 4 Realizar un modelo predictivo de la variable impago

In [77]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

## Analisis del CSV y selección justificada de las metricas de evaluacion para el modelo

In [78]:
df = pd.read_csv("Data/df_filtrado.csv")

In [79]:
df.head()

,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,...,Fiador,Impago,Prima,Tipo_Jornada_Laboral_Autónomo,Tipo_Jornada_Laboral_Desempleado,Tipo_Jornada_Laboral_Jornada completa,Tipo_Jornada_Laboral_Tiempo parcial,Estado_Civil_Casado,Estado_Civil_Divorciado,Estado_Civil_Soltero
0,6B4H6E,22,16000.0,10000,769,22.0,1,8.96,48.0,0.32,...,0.0,0,65.85,0,1,0,0,0,0,1
1,VKTUXP,33,38308.0,16474,699,70.0,3,14.07,36.0,0.20,...,0.0,0,260.60,0,0,1,0,1,0,0
2,5P9OVL,51,51271.0,40770,812,309.0,1,12.66,12.0,0.36,...,1.0,0,221.02,0,0,1,0,1,0,0
3,YL0W8G,30,31923.0,36707,813,43.0,1,8.96,24.0,0.32,...,0.0,0,125.00,0,0,1,0,1,0,0
4,YP8WTV,28,27373.0,86981,765,60.0,2,10.07,48.0,0.55,...,0.0,0,800.00,0,0,1,0,0,0,1


In [80]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 101886 entries, 0 to 101885
Data columns (total 23 columns):
 #   Column                                 Non-Null Count   Dtype  
---  ------                                 --------------   -----  
 0   ID                                     101886 non-null  str    
 1   Edad                                   101886 non-null  int64  
 2   Ingresos                               101886 non-null  float64
 3   Monto_Inicial                          101886 non-null  int64  
 4   Scoring_Crediticio                     101886 non-null  int64  
 5   Meses_Empleo                           101886 non-null  float64
 6   Num_Creditos                           101886 non-null  int64  
 7   Ratio_Interes                          101886 non-null  float64
 8   Duracion                               101886 non-null  float64
 9   Ratio_Deuda_Ingresos                   101886 non-null  float64
 10  Estudios                               101886 non-null  int64  
 11

Lo siguiente va a ser ver la correlacion entre las variables , ya que si hay una correlacion alta entre estas fallariamos uno de los supuestos para la regresion logistica y por lo tanto nuestro modelo tendria multicolinealidad

In [81]:
df = df.drop(columns="ID")

In [82]:
corr = df.corr().abs().stack()
corr[(corr > 0.7) & (corr < 1)]

Edad          Meses_Empleo    0.941027
Meses_Empleo  Edad            0.941027
dtype: float64

Lo que se ha hecho en la celda anterior ha sido crear una matriz de correlacion, al haber tantas variables hemos optado por pasar esa matriz a una lista. Además he pasado la correlacion a valor absoluto porque puede aver tanto correlacion postiva como correlacion negativa. Finalmente, hemos pedido que nos devuelva solo las variables en las que su correlacion es mayor a 0,70. El resultado ha sido que la variable meses_empleo y edad tienen una fuerte correlacion positiva (0.94).

Como tenemos que seleccionar una y ambas son bastante parecidas voy a optar con quedarme con Edad porque es una variable más sencilla de explicar

In [83]:
df = df.drop(columns="Meses_Empleo")

Como este dataset ya ha sido filtrado, ni tenemos nulos ni variables categoricas, por lo que podemos seguir viendo la viariable impago.

Aun asi viendo las variables me he dado cuenta que a la hora de hacer un get dummies de las variables categoricas se nos olvido eliminar una columna por lo que vamos a eliminar una de cada una. Esto es necesario porque de no hacerlo la suma de las categorias siempre sera 1 porque imaginate si tengo tres categorias (en formato dummy) y se dos de ellas no necesito la tercera 

Para ello he decidido eliminar la mas frecuente en ambas para que sea una referencia intuitiva

Primero vamos a elegir las columnas de cada categoria

In [91]:
cols_estado_civil = ["Estado_Civil_Casado", "Estado_Civil_Divorciado", "Estado_Civil_Soltero"]
cols_jornada = ["Tipo_Jornada_Laboral_Autónomo", "Tipo_Jornada_Laboral_Desempleado",
                "Tipo_Jornada_Laboral_Jornada completa", "Tipo_Jornada_Laboral_Tiempo parcial"]

In [ ]:
estado_civil_media = df[cols_estado_civil].mean()
print(estado_civil_media)

Estado_Civil_Casado        0.511778
Estado_Civil_Divorciado    0.186424
Estado_Civil_Soltero       0.301798
dtype: float64


In [95]:
df= df.drop(columns="Estado_Civil_Casado")

In [96]:
jornada_media = df[cols_jornada].mean()
print(jornada_media)

Tipo_Jornada_Laboral_Autónomo            0.148519
Tipo_Jornada_Laboral_Desempleado         0.099513
Tipo_Jornada_Laboral_Jornada completa    0.550056
Tipo_Jornada_Laboral_Tiempo parcial      0.201912
dtype: float64


In [97]:
df= df.drop(columns="Tipo_Jornada_Laboral_Jornada completa")

Ahora procedemos a analizar la variable objetivo

In [84]:
df["Impago"].value_counts()

Impago
0    90158
1    11728
Name: count, dtype: int64

In [85]:
df["Impago"].value_counts(normalize=True) * 100

Impago
0    88.489096
1    11.510904
Name: proportion, dtype: float64

La variable impago esta desbalanceada claramente esto nos ayuda a definir la metricas de evaluacion. En el 88.48 % de los casos el prestamo se paga, por lo que el modelo sin aprender nada simplemente diciendo que todas se pagan tendria un 88.49 % de acuraccy (de todos los datos cuantos he acertado). Esto nos dice que la metrica de evaluacion acuraccy puede llegar a ser engañosa. Por ello, vamos a usar esta metrica solo para compararla con este 88.48% que tendria si el modelo no aprendiese.

Por otro lado, vamos centrarnos en las metricas Precision y Recall:

Recall: de los todos los positivos, cuatos detectas.

Precision: de los que detectas como positivos cuantos lo son de verdad.

Ademas vamos a dar mas importancia al Recall porque consideramos que el coste de no detectar un impago es mas alto para la aseguradora, que el de marcar a una persona marcada como posible no pagador y que luego finalmente termine pagando





## Separar los datos en tres conjuntos (TRAIN, TEST y PUBLICO)

A continuacion vamos a separar los datos en tres conjunto Train, Test y Publico. Con Train vamos a entrenar el modelo. Con test vamos a validar ese modelo. Por ultimo con publico (que va a ser una muestra sacada del df desde el principio y no se va a tocar en ningun momento), vamos a simular el rendimiento de nuestro modelo cuando salga al los datos publicos

In [ ]:
df_modelo = df.copy()

#cogemos el 10% de los datos de nuestro df 
df_publico = df_modelo.sample(frac=0.10, random_state=42) #la funcion sample nos permite coger filas al azar ademas con random_state 42 creamos una semilla para que siempre coja las mismas filas que se seleccionaron la primera vez de su ejecucion

df_modelo = df_modelo.drop(df_publico.index) #ahora vamos a quitar del df modelo las filas que hemos seleccionado con el sample 

In [87]:
df_publico["Impago"].value_counts(normalize= True)*100

Impago
0    88.850721
1    11.149279
Name: proportion, dtype: float64

Vemos que en el conjunto publico se mantiene el desbalance que existe en la variable impago, un 11,14 % de impagos, por lo que podemos seguir separando en train test

Vamos a separar ahora si en train y test con la variable stratify para que los datos de la variable impago mantengan el desbalance, además añadimos la funcion shuffle para asegurarnos la aleatoriedad de la muestra


In [88]:
y = df_modelo["Impago"]
X = df_modelo.drop(columns="Impago")

In [89]:
X_train, X_test, y_train, y_test = train_test_split(X,y,stratify=df_modelo["Impago"], shuffle=True, random_state=42, train_size= 0.8)

Ahora vamos a ver los valores de x_train

In [90]:
X_train.describe()

,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,Estudios,Posesion_Hipoteca,Personas_Cargo,Fiador,Prima,Tipo_Jornada_Laboral_Autónomo,Tipo_Jornada_Laboral_Desempleado,Tipo_Jornada_Laboral_Jornada completa,Tipo_Jornada_Laboral_Tiempo parcial,Estado_Civil_Casado,Estado_Civil_Divorciado,Estado_Civil_Soltero
count,73357.000000,73357.000000,73357.000000,73357.000000,73357.000000,73357.000000,73357.00000,73357.000000,73357.000000,73357.000000,73357.000000,73357.000000,73357.000000,73357.000000,73357.000000,73357.000000,73357.000000,73357.000000,73357.000000,73357.000000
mean,39.645119,41465.166037,45050.881838,692.273471,1.995924,10.678138,37.27677,0.321583,0.927014,0.352441,0.449010,0.297150,385.251452,0.146803,0.098995,0.552149,0.202053,0.512126,0.186499,0.301375
std,10.107522,16027.314384,46681.109694,87.297816,0.995846,3.796769,14.18228,0.181365,0.927105,0.477734,0.497397,0.457006,267.458881,0.353911,0.298658,0.497276,0.401534,0.499856,0.389511,0.458859
min,19.000000,16000.000000,3000.000000,398.000000,1.000000,2.010000,12.00000,0.100000,0.000000,0.000000,0.000000,0.000000,10.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,33.000000,30437.000000,14978.000000,636.000000,1.000000,7.970000,24.00000,0.140000,0.000000,0.000000,0.000000,0.000000,137.530000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,39.000000,39059.000000,40000.000000,704.000000,2.000000,10.600000,36.00000,0.290000,1.000000,0.000000,0.000000,0.000000,336.970000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000
75%,46.000000,50123.000000,47534.000000,760.000000,3.000000,13.260000,48.00000,0.550000,1.000000,1.000000,1.000000,1.000000,614.230000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000
max,64.000000,120000.000000,400000.000000,845.000000,4.000000,24.440000,60.00000,0.550000,3.000000,1.000000,1.000000,1.000000,800.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
